# Validacion de varianza en dataset

In [ ]:
import os
import random
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from skimage.metrics import structural_similarity as ssim
from itertools import combinations, product


# CONFIGURACIÓN DE RUTAS Y PARÁMETROS
TRAIN_DIR = r'C:\Users\juanj\OneDrive\Documentos\ITESO\Cuarto Semestre\IDI IV\IDI III\idi_class_repo\data\processed\image\balanced_image\training\bcc'
TEST_DIR = r'C:\Users\juanj\OneDrive\Documentos\ITESO\Cuarto Semestre\IDI IV\IDI III\idi_class_repo\data\processed\image\balanced_image\testing\bcc'

SAMPLE_SIZE = 200  # Cantidad de imágenes a muestrear 
IMAGE_SIZE = (224, 224)

# ==========================================
# PREPARACIÓN DEL MODELO EXTRACTOR (ResNet50)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet50_Weights.DEFAULT
extractor = resnet50(weights=weights).to(device)
extractor.eval()
extractor = torch.nn.Sequential(*list(extractor.children())[:-1])

transform_cnn = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def load_images_and_embeddings(directory, sample_size):
    all_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(('.jpg', '.png'))]
    sampled_files = random.sample(all_files, min(sample_size, len(all_files)))
    
    images_gray = []
    embeddings = []
    
    with torch.no_grad():
        for file in sampled_files:
            img = Image.open(file).convert('RGB')
            
            # Para SSIM (escala de grises)
            img_gray = np.array(img.resize(IMAGE_SIZE).convert('L'))
            images_gray.append(img_gray)
            
            # Para Similitud del Coseno (Embeddings)
            img_tensor = transform_cnn(img).unsqueeze(0).to(device)
            emb = extractor(img_tensor).flatten()
            embeddings.append(emb)
            
    return images_gray, torch.stack(embeddings)

def calculate_similarity(images, embeddings, compare_pairs):
    ssim_scores = []
    cosine_scores = []
    cos_sim = torch.nn.CosineSimilarity(dim=0)
    
    for i, j in compare_pairs:
        # SSIM
        score, _ = ssim(images[i], images[j], full=True, data_range=255)
        ssim_scores.append(score)
        
        # Similitud Coseno
        cos_score = cos_sim(embeddings[i], embeddings[j]).item()
        cosine_scores.append(cos_score)
        
    return np.mean(ssim_scores), np.mean(cosine_scores)


print(f"Cargando {SAMPLE_SIZE} imágenes de entrenamiento...")
train_imgs, train_embs = load_images_and_embeddings(TRAIN_DIR, SAMPLE_SIZE)

print(f"Cargando {SAMPLE_SIZE} imágenes de prueba...")
test_imgs, test_embs = load_images_and_embeddings(TEST_DIR, SAMPLE_SIZE)

# 1. Similitud Intra-set (Variedad de datos)
train_pairs = list(combinations(range(len(train_imgs)), 2))
test_pairs = list(combinations(range(len(test_imgs)), 2))

train_ssim, train_cos = calculate_similarity(train_imgs, train_embs, train_pairs)
test_ssim, test_cos = calculate_similarity(test_imgs, test_embs, test_pairs)

# 2. Similitud Inter-set (Comprobación de Fuga de Datos)
# Comparamos cada imagen de entrenamiento con cada imagen de prueba
ssim_cross_scores = []
cosine_cross_scores = []
cos_sim = torch.nn.CosineSimilarity(dim=0)

for i, j in product(range(len(train_imgs)), range(len(test_imgs))):
    score, _ = ssim(train_imgs[i], test_imgs[j], full=True, data_range=255)
    ssim_cross_scores.append(score)
    cos_score = cos_sim(train_embs[i], test_embs[j]).item()
    cosine_cross_scores.append(cos_score)

cross_ssim = np.mean(ssim_cross_scores)
cross_cos = np.mean(cosine_cross_scores)

# resuultados
print("\n" + "="*50)
print(" REPORTE DE VARIANZA Y SIMILITUD DE DATOS")
print("="*50)
print(f"Métrica SSIM (0 = Diferentes, 1 = Clones exactos)")
print(f"Métrica Coseno (0 = Sin relación, 1 = Características idénticas)\n")

print(f"1. Varianza en Entrenamiento (Train vs Train):")
print(f"   - SSIM Promedio: {train_ssim:.4f}")
print(f"   - Similitud Coseno Promedio: {train_cos:.4f}")

print(f"\n2. Varianza en Prueba (Test vs Test):")
print(f"   - SSIM Promedio: {test_ssim:.4f}")
print(f"   - Similitud Coseno Promedio: {test_cos:.4f}")

print(f"\n3. Comprobación de Fuga (Train vs Test):")
print(f"   - SSIM Promedio: {cross_ssim:.4f}")
print(f"   - Similitud Coseno Promedio: {cross_cos:.4f}")
print("="*50)

# EfficientNetV2-S

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_V2_S_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Importar las métricas de evaluación adicionales
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# transformaciones para imagen
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# paths
train_path = r'C:\Users\juanj\OneDrive\Documentos\ITESO\Cuarto Semestre\IDI IV\IDI III\idi_class_repo\data\processed\image\balanced_image\training'
test_path = r'C:\Users\juanj\OneDrive\Documentos\ITESO\Cuarto Semestre\IDI IV\IDI III\idi_class_repo\data\processed\image\balanced_image\testing'

train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_path, transform=test_transform)

class_names = train_dataset.classes
print("Clases encontradas:", class_names)
print("Mapeo clases->idx:", train_dataset.class_to_idx)

# cargar datos
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Total imágenes entrenamiento: {len(train_dataset)}")
print(f"Total imágenes test: {len(test_dataset)}")

# definir modelo
efficientnet = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
num_classes = len(class_names)

# Ajustamos el clasificador de EfficientNetV2-S (recibe 1280 features de salida)
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(1280, 4096),
    nn.ReLU(True),
    nn.Dropout(p=0.5),
    nn.Linear(4096, 4096),
    nn.ReLU(True),
    nn.Linear(4096, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")
efficientnet.to(device)

# optimizador y función de pérdida (loss)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

# funciones train y test
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch} (Train)")
    for data, target in progress:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(1)
        correct += pred.eq(target).sum().item()

        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / len(train_loader.dataset)

    print(f"Epoch {epoch} -> Train Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy


def test(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    preds = []
    targets_list = []

    with torch.no_grad():
        progress = tqdm(test_loader, desc="Testing")
        for data, target in progress:
            data, target = data.to(device), target.to(device)

            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()

            pred = output.argmax(1)
            correct += pred.eq(target).sum().item()

            preds.extend(pred.cpu().numpy())
            targets_list.extend(target.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / len(test_loader.dataset)

    print(f"Test -> Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy, preds, targets_list

# entrenamiento
num_epochs = 5

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

best_acc = 0
best_preds = []
best_targets = []
# Se actualizó el nombre del archivo final para reflejar el uso de EfficientNet
best_model_path = r'C:\Users\juanj\OneDrive\Documentos\ITESO\Cuarto Semestre\IDI IV\IDI III\idi_class_repo\models\skin_cancer_efficientnet_model_final.pth'

# early stop
patience = 5
best_loss = float("inf")
counter = 0

print("Iniciando entrenamiento...")

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train(efficientnet, device, train_loader, criterion, optimizer, epoch)
    test_loss, test_acc, preds, targets_list = test(efficientnet, device, test_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    scheduler.step(test_loss)

    # early stopping
    if test_loss < best_loss:
        best_loss = test_loss
        counter = 0
    else:
        counter += 1
        print(f"EarlyStopping: {counter}/{patience} épocas sin mejora.")
        if counter >= patience:
            print("Early Stopping activado: no hubo mejora en test loss.")
            break

    # guardar mejor accuracy
    if test_acc > best_acc:
        best_acc = test_acc
        os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
        torch.save(efficientnet.state_dict(), best_model_path)
        best_preds = preds
        best_targets = targets_list
        print(f"Nuevo mejor modelo con accuracy: {best_acc:.2f}%. Guardado.")

    print("-" * 50)

print("Entrenamiento terminado.")
print(f"Mejor accuracy en test: {best_acc:.2f}%")

# graficas
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["test_loss"], label="Test Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train Acc")
plt.plot(epochs_range, history["test_acc"], label="Test Acc")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# métricas de evaluación adicionales y matriz de confusión
if best_preds:
    print("\n" + "="*50)
    print("MÉTRICAS DE EVALUACIÓN DEL MEJOR MODELO")
    print("="*50)
    
    # Reporte de clasificación (F1, Precision, Recall, Support)
    report = classification_report(best_targets, best_preds, target_names=class_names)
    print(report)

    # Matriz de confusión
    cm = confusion_matrix(best_targets, best_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    fig, ax = plt.subplots(figsize=(10, 10))
    disp.plot(cmap=plt.cm.Blues, xticks_rotation='vertical', ax=ax)
    plt.title("Confusion Matrix (Best Model)")
    plt.tight_layout()
    plt.show()
else:
    print("No se generaron métricas ni matriz de confusión (no se guardó modelo).")